# Auliya B→C — RGB vs SHAPECOEF1 (Kaggle)
Bandingkan YOLO26n-seg RGB dengan SHAPECOEF1 dari bobot pretrained resmi yang sama. Seed 42, 50 epoch, 640 px, batch 8; train B (375), validation C (420). **Group A tidak dibuka.** Ini bukan protokol YOLO26s dari notebook Semantic Transfer.

Pasang kode/bobot `auliya-shape-coeff-kaggle-code-v1.zip` sebagai private Kaggle Dataset (ZIP atau hasil ekstraknya). Pasang juga Dataset B/C yang sudah ada, atau `auliya-robusta-sni-b-train-c-val.zip`. GPU T4 untuk training; Internet ON untuk dependensi dan aset runtime Ultralytics. Tidak ada `git clone` atau login GitHub dari notebook.

Jalankan dengan `AUTHORIZE_TRAINING=False` untuk audit. Setelah PASS, jalankan sel training dengan `True` bila siap. Untuk dua sesi, set `RUN_ARMS` ke satu arm pada masing-masing sesi dan gunakan tipe GPU yang sama.

Simpan **Notebook Output lengkap** untuk resume, lalu tambahkan output versi tersebut sebagai input pada sesi berikutnya. `*-reports.zip` berisi laporan/log saja. Snapshot yang belum tersimpan oleh Kaggle tidak bisa dipulihkan. Jika ada lebih dari satu versi, isi `PREVIOUS_OUTPUT` dengan path folder run yang dipilih.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
STAGE = Path('/tmp/auliya-shape-coeff-kaggle-v1')
OUT = WORK / 'auliya-bc-shape-coeff-v1'
DATA_OVERRIDE = None  # folder development atau ZIP B/C, bila autodiscovery ambigu
PREVIOUS_OUTPUT = None  # folder auliya-bc-shape-coeff-v1 dari Notebook Output versi sebelumnya

MANIFEST = 'shape_coeff_bundle_manifest.json'
MANIFEST_SHA = 'ef51a83954b498a1ffbd5c80576a584b54a814b6028015714d389aefc6640cb2'
BUNDLE_ZIP = 'auliya-shape-coeff-kaggle-code-v1.zip'
zips, manifests = sorted(INPUT.rglob(BUNDLE_ZIP)), sorted(INPUT.rglob(MANIFEST))
if len(zips) + len(manifests) != 1:
    raise FileNotFoundError(f'Pasang tepat satu paket kode SHAPECOEF1; ZIP={zips}, manifest={manifests}')

def install_payload(read):
    raw = read(MANIFEST)
    if hashlib.sha256(raw).hexdigest() != MANIFEST_SHA:
        raise ValueError('Manifest kode tidak cocok dengan notebook ini')
    manifest = json.loads(raw)
    for path, digest in manifest['files'].items():
        content = read(path)
        if hashlib.sha256(content).hexdigest() != digest:
            raise ValueError(f'SHA256 paket kode tidak cocok: {path}')
        dest = STAGE / 'bundle' / path
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(content)
    print('KODE BASE:', manifest['base_commit'], '| BUNDLE MANIFEST SHA256:', MANIFEST_SHA)

if zips:
    with zipfile.ZipFile(zips[0]) as z:
        install_payload(z.read)
else:
    install_payload(lambda name: (manifests[0].parent / name).read_bytes())

spec = importlib.util.spec_from_file_location('shape_kaggle', STAGE / 'bundle/tools/auliya_shape_coeff_kaggle.py')
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
print('Memeriksa SHA256 input B/C...', flush=True)
DATA = helper.stage_data(INPUT, STAGE / 'development', helper.read_json(STAGE / 'bundle/development_files.json'), DATA_OVERRIDE)
WEIGHTS = STAGE / 'bundle/weights/yolo26n-seg.pt'
print('B/C:', DATA, '| WEIGHTS SHA256:', helper.sha(WEIGHTS), '| TEST: False')

ENV = os.environ.copy()
(STAGE / 'ultralytics').mkdir(exist_ok=True)
ENV.update(PYTHONPATH=str(STAGE / 'bundle/src'),
           AULIYA_KAGGLE_BUNDLE_SHA256=MANIFEST_SHA, YOLO_CONFIG_DIR=str(STAGE / 'ultralytics'))
check_pin = [sys.executable, '-c', "from importlib.metadata import version; raise SystemExit(version('ultralytics') != '8.4.96')"]
if subprocess.run(check_pin, env=ENV, capture_output=True).returncode:
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96'], env=ENV, capture_output=True, text=True)
    if install.returncode:
        raise RuntimeError(f'Gagal memasang Ultralytics 8.4.96. Cek Internet Kaggle.\n{install.stderr[-2500:]}')
probe = "import os; from pathlib import Path; import torch; import ultralytics; import coffee_detector; assert ultralytics.__version__ == '8.4.96'; assert Path(coffee_detector.__file__).resolve().is_relative_to(Path(os.environ['PYTHONPATH']).resolve()); print('VERSION:', ultralytics.__version__, torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); print('SOURCE:', coffee_detector.__file__); print('DEVICE', '0' if torch.cuda.is_available() else 'cpu')"
runtime = subprocess.run([sys.executable, '-c', probe], env=ENV, capture_output=True, text=True)
if runtime.returncode:
    raise RuntimeError(runtime.stderr or runtime.stdout)
print(runtime.stdout.strip())
DEVICE = next(line.split()[1] for line in runtime.stdout.splitlines() if line.startswith('DEVICE '))
COMMAND = [sys.executable, '-u', '-m', 'coffee_detector.experiments.run_auliya_shape_coeff',
           '--development', str(DATA), '--weights', str(WEIGHTS), '--output', str(OUT), '--device', DEVICE]


In [ ]:
helper.run_logged(COMMAND, ENV, OUT)
audit = helper.read_json(OUT / 'static_audit.json')
if audit['decision'] != 'PASS' or audit['test_images_accessed'] is not False:
    raise RuntimeError(f'STOP: audit tidak lolos: {audit}')
print(json.dumps(audit, indent=2))


In [ ]:
AUTHORIZE_TRAINING = False  # ubah ke True sendiri setelah audit GPU PASS
RUN_ARMS = ('RGB', 'SHAPECOEF1')
if AUTHORIZE_TRAINING:
    if DEVICE != '0' or audit['device'] != 'cuda:0' or audit['decision'] != 'PASS':
        raise RuntimeError('STOP: pilih Accelerator GPU T4 dan jalankan ulang setup + audit GPU.')
    if not RUN_ARMS or len(set(RUN_ARMS)) != len(RUN_ARMS) or not set(RUN_ARMS).issubset(helper.ARMS):
        raise ValueError(f'RUN_ARMS tidak valid: {RUN_ARMS}')
    contract_code = "import json; from coffee_detector.experiments.run_auliya_shape_coeff import run_contract; print('CONTRACT ' + json.dumps({a: run_contract(a, '0') for a in ('RGB', 'SHAPECOEF1')}))"
    p = subprocess.run([sys.executable, '-c', contract_code], env=ENV, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr or p.stdout)
    contracts = json.loads(next(line[9:] for line in p.stdout.splitlines() if line.startswith('CONTRACT ')))
    helper.restore_arms(INPUT, OUT, contracts, PREVIOUS_OUTPUT)
    for arm in RUN_ARMS:
        helper.run_logged(COMMAND + ['--arm', arm, '--authorize-training'], ENV, OUT, arm)
        print('SELESAI:', arm, '|', OUT / arm / 'validation_result.json', flush=True)
else:
    print('AUDIT ONLY | TRAINING: False | TEST: False')


In [ ]:
helper.summarize(OUT)
helper.report_zip(OUT)
